In [0]:

# Tamaño de la información cargada.
lineas = spark.sql("""
    SELECT
        COUNT(*) AS filas,
        COUNT(DISTINCT source_file) AS archivos_con_datos
    FROM workspace.default.bronze_pos_raw
""").first()

# Resumen de pedidos e importes.
pedidos = spark.sql("""
    SELECT
        COUNT(*) AS pedidos,
        COUNT(DISTINCT store_number) AS tiendas,
        COUNT(DISTINCT fecha) AS fechas_con_datos,
        MIN(fecha) AS fecha_inicial,
        MAX(fecha) AS fecha_final,
        COUNT(importe_comparable) AS pedidos_con_importe,
        COUNT(*) - COUNT(importe_comparable) AS pedidos_sin_importe,
        ROUND(
            100.0 * COUNT(importe_comparable) / NULLIF(COUNT(*), 0),
            2
        ) AS cobertura_pct,
        ROUND(SUM(importe_comparable), 2) AS importe_estimado
    FROM workspace.default.silver_orders
""").first()

print("RESUMEN DE LA BASE COMPLETA")
print(f"Filas originales cargadas: {lineas['filas']:,}")
print(f"Archivos con datos: {lineas['archivos_con_datos']:,}")
print(f"Pedidos: {pedidos['pedidos']:,}")
print(f"Tiendas: {pedidos['tiendas']:,}")
print(f"Fechas con datos: {pedidos['fechas_con_datos']:,}")
print(f"Primera fecha: {pedidos['fecha_inicial']}")
print(f"Última fecha: {pedidos['fecha_final']}")
print(f"Pedidos con importe: {pedidos['pedidos_con_importe']:,}")
print(f"Pedidos sin importe: {pedidos['pedidos_sin_importe']:,}")
print(f"Cobertura del importe: {pedidos['cobertura_pct']}%")
print(f"Suma del importe estimado: {pedidos['importe_estimado']}")

In [0]:
mensual = spark.table("workspace.default.gold_monthly_kpis")

display(mensual.orderBy("mes"))

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Solo llevamos a pandas el resumen mensual.
datos = (
    spark.table("workspace.default.gold_monthly_kpis")
    .orderBy("mes")
    .toPandas()
)

datos["mes"] = pd.to_datetime(datos["mes"])

graficas = [
    (
        "pedidos_por_tienda_dia",
        "Pedidos por tienda-día con actividad",
        "Pedidos",
        "#2563EB"
    ),
    (
        "importe_promedio_por_pedido",
        "Importe promedio de pedidos con estimación",
        "Importe",
        "#059669"
    ),
    (
        "pedidos_con_descuento_pct",
        "Pedidos con descuento informado",
        "Porcentaje de pedidos",
        "#D97706"
    ),
    (
        "cobertura_importe_pct",
        "Pedidos con importe comparable",
        "Porcentaje de pedidos",
        "#7C3AED"
    )
]

fig, ejes = plt.subplots(2, 2, figsize=(15, 9))

for eje, (columna, titulo, etiqueta, color) in zip(
    ejes.flat, graficas
):
    valores = datos[columna].astype(float)

    eje.plot(
        datos["mes"],
        valores,
        marker="o",
        markersize=4,
        linewidth=2,
        color=color
    )

    eje.set_title(titulo, fontsize=12)
    eje.set_ylabel(etiqueta)
    eje.set_xlabel("Mes")
    eje.grid(axis="y", alpha=0.25)

    eje.xaxis.set_major_locator(
        mdates.MonthLocator(interval=3)
    )
    eje.xaxis.set_major_formatter(
        mdates.DateFormatter("%m/%Y")
    )
    eje.tick_params(axis="x", rotation=45)

    if columna == "cobertura_importe_pct":
        eje.set_ylim(95, 100)
    else:
        eje.set_ylim(bottom=0)

fig.suptitle(
    "Taco Bueno · Indicadores mensuales del pipeline",
    fontsize=16
)

fig.text(
    0.5,
    0.02,
    "Las tiendas y fechas disponibles varían entre meses. "
    "Los días sin registros no se cuentan como cero.",
    ha="center",
    fontsize=10
)

plt.tight_layout(rect=[0, 0.06, 1, 0.94])
plt.show()